# **GET EMBEDDINGS USING GOOGLE COLAB (GPU)**

In [1]:
# set up mounting gdrive and accessing env data.
from google.colab import drive
from google.colab import userdata
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/ml_projects/data/complaints_data"
CACHE_DIR = userdata.get('SENTENCE_TRANSFORMERS_HOME')

Mounted at /content/drive


In [2]:
# load libraries
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer

In [3]:
# load model + data

# load data
data_files = {"train":DATA_DIR+"/train.csv",
              "validation":DATA_DIR+"/val.csv"}
raw_dataset = load_dataset("csv", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [4]:
# clean complaint text before generating embeddings.

def clean_text(sentence):
    sentence = sentence.lower()
    # remove punctuations
    sentence = re.sub(r"([!\"'#$%&()*\+-/:;<=>?@\\\[\]^_`{|}~])", r" \1 ", sentence)
    # remove non-alphanumeric.
    sentence = re.sub("[^A-Za-z0-9]+", " ", sentence)
    # remove repeated XXXX characters
    pattern = re.compile(r"(xx+\s*)+")
    # rename repeated XXXX characters to 'mask'
    sentence = pattern.sub("mask ", sentence)
    sentence = re.sub(" +", " ", sentence).strip() # remove repeated spaces
    sentence = " ".join(sentence.split())
    sentence = re.sub(r"http\S+", "", sentence) # remove hyperlinks
    return sentence

In [5]:
# apply clean_text function to complaint text (in batches).
raw_dataset = raw_dataset.map(lambda x:{"text": [clean_text(text) for text in x['complaint_what_happened']]}, batched=True)

Map:   0%|          | 0/14198 [00:00<?, ? examples/s]

Map:   0%|          | 0/3550 [00:00<?, ? examples/s]

In [6]:
# create a new column that has the length of each complaint.
# this is to filter complaints that have extremely short sentences or sentences that only contain the pattern :XXXX ...

raw_dataset = raw_dataset.map(lambda x: {"text_length": [len(text.split())for text in x['text']]}, batched=True)

Map:   0%|          | 0/14198 [00:00<?, ? examples/s]

Map:   0%|          | 0/3550 [00:00<?, ? examples/s]

In [7]:
# filtering sentences with more than 6 words.
raw_dataset = raw_dataset.filter(lambda x: x['text_length']>6)

Filter:   0%|          | 0/14198 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3550 [00:00<?, ? examples/s]

In [8]:
# Get label names and map them to index. This will be the y values when training
label2idx = {label_name: idx for idx, label_name in enumerate(sorted(raw_dataset['train'].unique("labels")))}
print(f"Label Mappings: {label2idx}")


Flattening the indices:   0%|          | 0/14192 [00:00<?, ? examples/s]

Label Mappings: {'Banking Operations': 0, 'Cards & Payments': 1, 'Collections & Recovery': 2, 'Consumer Lending': 3, 'Credit Reporting & Disputes': 4, 'Money Transfer and Payments': 5, 'Mortgage & Home Lending': 6}


In [9]:
# get index.
raw_dataset = raw_dataset.map(lambda x: {"labels": label2idx[x['labels']]})

Map:   0%|          | 0/14192 [00:00<?, ? examples/s]

Map:   0%|          | 0/3550 [00:00<?, ? examples/s]

In [10]:
# change format of dataset tp pandas

raw_dataset.set_format("pandas")
train_complaints_df = raw_dataset['train'][:][["product", 'complaint_what_happened', 'labels', "text"]].copy()
val_complaints_df = raw_dataset['validation'][:][["product", 'complaint_what_happened', 'labels', "text"]].copy()

### **Load embeddings model**: all-MiniLM-L6-v2

In [11]:
# load model
model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder=CACHE_DIR)

Loading weights:   0%|          | 0/103 [00:01<?, ?it/s]

In [12]:
# get embeddings for train & val

train_list = train_complaints_df.text.tolist()
val_list = val_complaints_df.text.tolist()

# get embeddings faster with cuda (GPU).
# embeddings are numpy array which will then be saved
train_complaints_embeddings = model.encode(train_list, batch_size=256, show_progress_bar=True, device='cuda')
val_complaints_embeddings = model.encode(val_list, batch_size=256, show_progress_bar=True, device='cuda')

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

In [13]:
# get labels for train and val set. These will be saved alongside embeddings.

y_train = train_complaints_df['labels'].values
y_val = val_complaints_df['labels'].values

In [15]:
# save data for further training.

train_path = DATA_DIR+"/train_embeddings.npy"
val_path = DATA_DIR+"/val_embeddings.npy"

with open(train_path, "wb") as f:
  np.save(f, train_complaints_embeddings)
  np.save(f, y_train)

with open(val_path, "wb") as f:
  np.save(f, val_complaints_embeddings)
  np.save(f, y_val)

## **END**